[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/MarcoLaud/VP2024/blob/main/VP2025/VP2025_Lecture2.ipynb)

# Linear regression

In this notebook, we will show how the gradient descent can determine the optimal parameters (m,q) of a line.

In the case under analysis, we will consider the relation between the hectolitre of Campari Soda drunk during the RES construction, and the RES performance.

We will see how a linear model can explain the data with a good approximation.

In [ ]:
# @title Start here
import pandas as pd

# Load the dataset
df = pd.read_csv("Pone_Secret_Data.csv")

In [ ]:
#@title Campari-RES regression
import matplotlib.pyplot as plt

# === Scatter plot of stress vs. strain ===
plt.figure(figsize=(8, 5))
plt.scatter(df['strain'], df['stress'], color='orange', alpha=0.7, edgecolors='k')
plt.xlabel("Campari [hl]")
plt.ylabel("Res performance [a.u.]")
plt.title("Campari vs RES performance")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#@title Set the learning rate
import ipywidgets as widgets
slider_log = widgets.FloatLogSlider(
    value=0.7,
    base=10,
    min=-1, # min exponent of base
    max=0.5, # max exponent of base
    step=0.05, # exponent step
    description='Log Slider'
)

display(slider_log)

In [ ]:
#@title Gradient descent demo
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# === Data ===
m = 20
theta0_true = 2
theta1_true = 0.5
x = np.linspace(-1, 1, m)
y = theta0_true + theta1_true * x + np.random.randn(m) * 0.2

def hypothesis(x, theta0, theta1):
    return theta0 + theta1 * x

def cost(theta0, theta1):
    return np.mean((hypothesis(x, theta0, theta1) - y) ** 2) / 2

# === Cost Surface ===
theta0_vals = np.linspace(-1, 4, 100)
theta1_vals = np.linspace(-5, 5, 100)
T0, T1 = np.meshgrid(theta0_vals, theta1_vals)
J_vals = np.vectorize(cost)(T0, T1)

# === Gradient Descent ===
N = 15
alpha = slider_log.value
theta0_initial = -0.9
theta1_initial = -4.2
theta_path = [np.array([theta0_initial, theta1_initial])]
J_path = [cost(theta0_initial, theta1_initial)]

for _ in range(N - 1):
    theta0, theta1 = theta_path[-1]
    pred = hypothesis(x, theta0, theta1)
    grad0 = np.mean(pred - y)
    grad1 = np.mean((pred - y) * x)
    new_theta = np.array([theta0 - alpha * grad0, theta1 - alpha * grad1])
    theta_path.append(new_theta)
    J_path.append(cost(*new_theta))

theta_path = np.array(theta_path)
J_path = np.array(J_path)

# === Frames for animation ===
frames = []
x_line = np.linspace(-1, 1, 100)

for i in range(1, len(theta_path) + 1):
    # Parameters at this step
    t0, t1 = theta_path[i - 1]
    y_pred = hypothesis(x_line, t0, t1)

    frames.append(go.Frame(
        name=f"step{i}",
        data=[
            # Surface
            go.Surface(x=theta0_vals, y=theta1_vals, z=J_vals,
                       colorscale='Viridis', opacity=0.8, showscale=False),
            # Descent path
            go.Scatter3d(
                x=theta_path[:i, 0],
                y=theta_path[:i, 1],
                z=J_path[:i],
                mode='lines+markers',
                line=dict(color='red', width=4),
                marker=dict(size=5, color='red')
            ),
            # Scatter of data
            go.Scatter(
                x=x, y=y, mode='markers',
                marker=dict(color='black', symbol='x', size=8),
                xaxis='x2', yaxis='y2'
            ),
            # Line fit at step i
            go.Scatter(
                x=x_line, y=y_pred,
                mode='lines',
                line=dict(color='red', width=3),
                xaxis='x2', yaxis='y2'
            )
        ]
    ))

# === Initial traces ===
t0, t1 = theta_path[0]
y_pred = hypothesis(x_line, t0, t1)

# Create subplots: 1 row, 2 columns
fig = make_subplots(rows=1, cols=2,
                    specs=[[{"type": "surface"}, {"type": "xy"}]],
                    column_widths=[0.6, 0.4],
                    subplot_titles=("Cost Function Surface", "Line Fit During Gradient Descent"))

# Add initial data
fig.add_trace(go.Surface(
    x=theta0_vals, y=theta1_vals, z=J_vals,
    colorscale='Viridis', opacity=0.8, showscale=False
), row=1, col=1)

fig.add_trace(go.Scatter3d(
    x=[theta_path[0, 0]],
    y=[theta_path[0, 1]],
    z=[J_path[0]],
    mode='lines+markers',
    marker=dict(size=5, color='red'),
    line=dict(color='red', width=4)
), row=1, col=1)

# Right plot: data + initial fit line
fig.add_trace(go.Scatter(
    x=x, y=y, mode='markers',
    marker=dict(color='black', symbol='x', size=8),
    xaxis='x2', yaxis='y2' # Ensure initial scatter is on x2/y2
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=x_line, y=hypothesis(x_line, *theta_path[0]),
    mode='lines',
    line=dict(color='red', width=3),
    xaxis='x2', yaxis='y2' # Ensure initial line is on x2/y2
), row=1, col=2)


# === Buttons ===
buttons = [
    dict(label="▶️ Play",
         method="animate",
         args=[None, {
             "frame": {"duration": 1000, "redraw": True},
             "fromcurrent": True,
             "transition": {"duration": 200}
         }]
    ),
    dict(label="⏸️ Pause",
         method="animate",
         args=[[None], {
             "mode": "immediate",
             "frame": {"duration": 0, "redraw": False},
             "transition": {"duration": 0}
         }]
    ),
    dict(label="🔁 Restart",
         method="animate",
         args=[[f"step1"], {
             "frame": {"duration": 0, "redraw": True},
             "mode": "immediate",
             "transition": {"duration": 0}
         }]
    )
]

# === Layout settings ===
fig.update_layout(
    title=dict(text="Gradient Descent: Surface + Line Fit", x=0.5, y=0.95),
    updatemenus=[dict(
        type="buttons",
        direction="left",
        showactive=False,
        buttons=buttons,
        x=0.5,
        y=1.02,
        xanchor="center",
        yanchor="bottom",
        pad={"r": 10, "t": 10, "b":25}
    )],
    margin=dict(t=120),
    width=1000,
    height=600,
    scene=dict(
        xaxis_title='θ₀',
        yaxis_title='θ₁',
        zaxis_title='Cost J(θ₀, θ₁)'
    ),
    # Explicitly set domain for x2 and y2 to constrain the second subplot
    xaxis2=dict(title="x", domain=[0.6, 1.0]),
    yaxis2=dict(title="y", domain=[0.0, 1.0])
)

# Add frames to figure
fig.frames = frames

fig.show()